# Chapitre 2 — Calculer avec des vecteurs et des matrices · Notebook de travail

*Maths pour l'IA, pas à pas*, Belk Siri — notebook compagnon.

[Ouvrir dans Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/fr/ch02_a_completer.ipynb) · Page compagnon : https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/

Les cellules à modifier portent la mention **À COMPLÉTER**. Exécutez les cellules dans l'ordre : tant qu'une réponse manque, les vérifications le signalent sans erreur. Le corrigé se trouve dans `ch02_corrige.ipynb`, sur la même page compagnon.

**Règle de prédiction du chapitre** (donnée, et non ajustée) : poids $\mathbf{w} = (3 ; -8)$ en k€/m² et k€/km, ordonnée à l'origine $b = 45$ k€.

Avec les quartiers (leçon 2.4), la règle devient $\mathbf{w} = (3 ; -8 ; 12 ; -10)$, $b = 45$ k€, la Gare étant le quartier de référence.

**Sommaire.** Données · Les calculs des leçons · Exercices 2.1 à 2.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Couleurs du livre : une couleur = un rôle (observé, prédit, écart, droite du collègue)
BLEU, ORANGE, ECART, VERT = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Données

Les identifiants (D2.1, D2.E2) sont ceux du livre et des fichiers CSV.

In [ ]:
# Données du chapitre 2, recopiées des fichiers CSV du livre (ne pas modifier)

# D2.1 — les sept logements avec leur distance au centre (leçons 2.1 à 2.3)
logements = pd.DataFrame({
    'logement':    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'surface_m2':  [32, 45, 54, 70, 85, 98, 62],
    'distance_km': [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'prix_kEUR':   [118, 172, 185, 245, 279, 335, 209],
})

# D2.4 — cinq logements de l'est de la ville, avec leur quartier (leçon 2.4)
est = pd.DataFrame({
    'logement':    ['K', 'L', 'M', 'N', 'O'],
    'quartier':    ['Parc', 'Rocade', 'Gare', 'Parc', 'Rocade'],
    'surface_m2':  [60, 60, 60, 45, 80],
    'distance_km': [2.0, 2.0, 2.0, 1.0, 3.0],
    'prix_kEUR':   [223, 200, 210, 186, 248],
})

# D2.5 — les cinq T2 du Sud avec leur distance au centre (leçon 2.5)
t2_sud = pd.DataFrame({
    'vente':       ['S03', 'S04', 'S05', 'S06', 'S07'],
    'surface_m2':  [43, 40, 44, 46, 52],
    'distance_km': [3.5, 2.0, 1.5, 2.5, 0.5],
})

# D2.E2 — trois logements mis en vente (exercice 2.2)
nouveaux = pd.DataFrame({
    'logement':    ['H', 'I', 'J'],
    'surface_m2':  [40, 75, 90],
    'distance_km': [0.5, 3.5, 1.0],
})

## Les calculs des leçons, en Python

Ces cellules refont les calculs des leçons 2.1 à 2.5. Exécutez-les et comparez avec le livre.

In [ ]:
# Leçon 2.1 — la règle de l'agence appliquée à G, terme par terme (équation 2.1)
part_surface = 3 * 62        # k€/m² × m² = k€
part_distance = -8 * 2.5     # k€/km × km = k€
part_fixe = 45               # k€
prix_G = part_surface + part_distance + part_fixe
print(part_surface, part_distance, part_fixe, '→', prix_G, 'k€ ; écart signé :', 209 - prix_G, 'k€')

In [ ]:
# Leçon 2.2 — vecteurs, produit scalaire (équation 2.2) et prédiction (équation 2.3)
x_G = np.array([62, 2.5])    # (surface en m², distance en km)
w = np.array([3, -8])        # (k€/m², k€/km), même ordre que x
b = 45                       # k€
print('dimension de x_G :', x_G.shape, '  w · x_G =', np.dot(w, x_G), '  ŷ =', np.dot(w, x_G) + b)
print('poids dans le mauvais ordre :', np.dot(w[::-1], x_G) + b, 'k€ (absurde)')

In [ ]:
# Leçon 2.3 — toutes les prédictions d'un coup (équation 2.5)
X = logements[['surface_m2', 'distance_km']].to_numpy(dtype=float)
y = logements['prix_kEUR'].to_numpy(dtype=float)
y_chapeau = X @ w + b
print('X :', X.shape, ' w :', w.shape, ' ŷ :', y_chapeau.shape)
print('ŷ =', y_chapeau)
print('écarts signés y − ŷ =', y - y_chapeau)
print('moyenne des écarts en valeur absolue, 7 logements :', round(np.mean(np.abs(y - y_chapeau)), 2), 'k€')
print('idem, A à F :', round(np.mean(np.abs(y - y_chapeau)[:6]), 2), 'k€ ; droite de l\'ouverture, A à F :',
      round(np.mean(np.abs(y - (3 * X[:, 0] + 30))[:6]), 2), 'k€')

In [ ]:
# Leçon 2.4 — colonnes indicatrices (équation 2.6), la Gare comme référence
indicatrices = pd.get_dummies(est['quartier'], drop_first=True).astype(int)   # colonnes Parc, Rocade
X_est = np.column_stack([est['surface_m2'], est['distance_km'], indicatrices['Parc'], indicatrices['Rocade']])
w_est = np.array([3, -8, 12, -10])
y_est = est['prix_kEUR'].to_numpy(dtype=float)
print(indicatrices.columns.tolist(), ' X :', X_est.shape)
print('avec le quartier :', X_est @ w_est + 45)
print('sans le quartier :', X_est[:, :2] @ w_est[:2] + 45)
print('moyenne des écarts en valeur absolue : sans', np.mean(np.abs(y_est - (X_est[:, :2] @ w_est[:2] + 45))),
      ' avec', np.mean(np.abs(y_est - (X_est @ w_est + 45))))

In [ ]:
# Leçon 2.5 — la même prédiction dans deux échelles (équation 2.7), division par n
X5 = t2_sud[['surface_m2', 'distance_km']].to_numpy(dtype=float)
mu, sigma = X5.mean(axis=0), X5.std(axis=0, ddof=0)
w, b = np.array([3, -8]), 45
w_std, b_std = w * sigma, b + np.sum(w * mu)
Z = (X5 - mu) / sigma
print('moyennes :', mu, ' écarts-types :', sigma)
print('poids standardisés :', w_std, ' ordonnée :', b_std)
print('unités d\'origine :', X5 @ w + b)
print('échelle standardisée :', Z @ w_std + b_std)

## Exercice 2.1 — Lire un schéma

*Notions : produit d'une matrice par un vecteur (leçon 2.3).* Reprenez le schéma matriciel de la leçon 2.3 (les sept logements A à G). La cellule ci-dessous reconstruit $X$, $\mathbf{w}$ et $b$ pour vérifier vos réponses.

In [ ]:
X = logements[['surface_m2', 'distance_km']].to_numpy(dtype=float)
w, b = np.array([3, -8]), 45
print(X)

In [ ]:
# À COMPLÉTER : vos réponses, lues sur le schéma
dimensions = {'X': None, 'w': None, 'y_chapeau': None}   # ex. (7, 2), (2,), (7,)
ligne_280 = None             # lettre du logement dont la prédiction vaut 280 k€
dimensions_c = {'X': None, 'w': None, 'y_chapeau': None}   # avec l'étage et un 8e logement

In [ ]:
# Vérification
y_chapeau = X @ w + b
vrai = {'X': X.shape, 'w': w.shape, 'y_chapeau': y_chapeau.shape}
if None in dimensions.values() or ligne_280 is None or None in dimensions_c.values():
    print('Complétez la cellule précédente.')
else:
    print('a) dimensions :', 'correct' if dimensions == vrai else 'à revoir')
    lettre = logements['logement'][int(np.argmin(np.abs(y_chapeau - 280)))]
    print('b) logement :', 'correct' if ligne_280 == lettre else 'à revoir')
    X8 = np.zeros((8, 3)); w3 = np.zeros(3)   # forme seulement : 8 logements, 3 caractéristiques
    print('c) dimensions :', 'correct' if dimensions_c == {'X': X8.shape, 'w': w3.shape,
          'y_chapeau': (X8 @ w3).shape} else 'à revoir')

*Vos explications :* …

## Exercice 2.2 — Trois nouveaux logements

*Données D2.E2. Notions : produit scalaire (leçon 2.2), produit d'une matrice par un vecteur (leçon 2.3).* Les questions a) à c) se font d'abord sur papier, avec le tableau de l'exercice 2.2 du livre.

In [ ]:
nouveaux

### a) et b) Sur papier

Écrivez les trois vecteurs, la matrice $X$ et ses dimensions, puis les trois prix prédits avec les unités. Reportez vos prix ci-dessous.

In [ ]:
# À COMPLÉTER : vos prix prédits calculés à la main, en k€
prix_main = {'H': None, 'I': None, 'J': None}

In [ ]:
# Vérification (calcul indépendant, ligne par ligne)
for _, l in nouveaux.iterrows():
    vrai = 3 * l['surface_m2'] - 8 * l['distance_km'] + 45
    la_votre = prix_main[l['logement']]
    etat = 'pas encore complété' if la_votre is None else ('correct' if np.isclose(la_votre, vrai) else 'à revoir')
    print(f"{l['logement']} : {etat}")

### c) Interpréter

*Votre réponse :* …

### d) Toutes les prédictions en une ligne

Complétez la cellule, puis exécutez la vérification.

In [ ]:
# X : une ligne par logement (H, I, J),
#     colonnes surface puis distance
X = None          # À COMPLÉTER
w = None          # À COMPLÉTER : ordre des colonnes de X
b = None          # À COMPLÉTER : en k€
y_chapeau = None  # À COMPLÉTER : équation 2.5

<details><summary><b>Indice</b> (cliquez si vous bloquez)</summary>

`nouveaux[['surface_m2', 'distance_km']].to_numpy()` donne la matrice. `np.array([..., ...])` crée un vecteur. Le produit matriciel s'écrit `@`.

</details>

In [ ]:
# Vérification
if y_chapeau is None:
    print('Complétez la cellule précédente.')
else:
    print('dimensions :', np.shape(X), '@', np.shape(w), '→', np.shape(y_chapeau))
    print('dimensions :', 'correct' if np.shape(X) == (3, 2) and np.shape(y_chapeau) == (3,) else 'à revoir')
    print('prédictions :', y_chapeau, '→',
          'correct' if np.allclose(y_chapeau, [161, 242, 307]) else 'à revoir')

### e) Prédire, puis vérifier

Colonnes de $X$ dans l'ordre distance puis surface, $\mathbf{w}$ inchangé. Notez d'abord votre prédiction du signe des trois prix, puis exécutez la cellule.

*Votre prédiction :* …

In [ ]:
if y_chapeau is not None:
    X_inverse = X[:, ::-1]          # colonnes échangées : distance, surface
    print('colonnes échangées :', X_inverse @ w + b)

## Exercice 2.3 — Que disent les poids ?

*Notions : lecture d'un poids (leçon 2.2), colonnes indicatrices (leçon 2.4).* Règle : $\mathbf{w} = (3 ; -8 ; 12 ; -10)$ pour surface, distance, Parc, Rocade ; $b = 45$ k€ ; la Gare est la référence. Questions sur papier, vérification ci-dessous.

In [ ]:
# À COMPLÉTER : écarts de prix prédits, en k€ (premier logement − second)
a_parc_moins_rocade = None
b_plus_loin_moins_proche = None   # 1,5 km plus loin, même quartier, même surface
c_parc_loin_moins_gare = None     # Parc et 1,5 km plus loin, comparé à la Gare

In [ ]:
# Vérification : on calcule la règle pour des paires de logements fictifs
regle = lambda surface, distance, parc, rocade: 3 * surface - 8 * distance + 12 * parc - 10 * rocade + 45
vrais = {'a': regle(60, 2, 1, 0) - regle(60, 2, 0, 1),
         'b': regle(60, 3.5, 0, 0) - regle(60, 2, 0, 0),
         'c': regle(60, 3.5, 1, 0) - regle(60, 2, 0, 0)}
for k, la_votre in [('a', a_parc_moins_rocade), ('b', b_plus_loin_moins_proche), ('c', c_parc_loin_moins_gare)]:
    etat = 'pas encore complété' if la_votre is None else ('correct' if np.isclose(la_votre, vrais[k]) else 'à revoir')
    print(f'{k}) {etat}')

*Vos interprétations pour c) et d) :* …

## Exercice 2.4 — Des colonnes indicatrices en Python

*Données D2.4. Notions : colonnes indicatrices (leçon 2.4), produit d'une matrice par un vecteur (leçon 2.3).*

In [ ]:
est

In [ ]:
# À COMPLÉTER
parc = None     # a) 1 si le logement est au Parc, 0 sinon (entiers)
rocade = None   # a) 1 si le logement est à la Rocade, 0 sinon
X4 = None       # b) 5 × 4 : surface, distance, parc, rocade
w4 = np.array([3, -8, 12, -10])
b4 = 45
y4 = None       # b) équation 2.5

<details><summary><b>Indice</b> (cliquez si vous bloquez)</summary>

`(est['quartier'] == 'Parc')` donne des vrai/faux ; `.astype(int)` les change en 1/0. `np.column_stack([a, b, c, d])` place quatre colonnes côte à côte.

</details>

In [ ]:
# Vérification
if y4 is None:
    print('Complétez la cellule précédente.')
else:
    print('colonnes indicatrices :', 'correct' if list(parc) == [1, 0, 0, 1, 0] and list(rocade) == [0, 1, 0, 0, 1] else 'à revoir')
    print('X4 :', np.shape(X4), '→', 'correct' if np.shape(X4) == (5, 4) else 'à revoir')
    print('prédictions :', y4, '→', 'correct' if np.allclose(y4, [221, 199, 209, 184, 251]) else 'à revoir')

**c)** Moyenne des écarts en valeur absolue, avec et sans les colonnes de quartier.

In [ ]:
# À COMPLÉTER
y_obs = est['prix_kEUR'].to_numpy(dtype=float)
eam_sans = None   # sans quartier : deux premières colonnes et deux premiers poids
eam_avec = None

In [ ]:
if eam_sans is None or eam_avec is None:
    print('Complétez la cellule précédente.')
else:
    print(f'sans : {eam_sans:.1f} k€, avec : {eam_avec:.1f} k€ →',
          'correct' if np.isclose(eam_sans, 10.2) and np.isclose(eam_avec, 1.8) else 'à revoir')

*Votre commentaire :* …

## Exercice 2.5 — Prédire, puis vérifier : changer d'unité

*Données D2.5. Notions : changement d'échelle (leçon 2.5, équation 2.7).* Notez vos prédictions pour a) et b) avant d'exécuter les cellules.

*Vos prédictions :* …

In [ ]:
# À COMPLÉTER : passage aux poids standardisés (équation 2.7)
def poids_standardises(w, b, mu, sigma):
    w_std = None   # À COMPLÉTER
    b_std = None   # À COMPLÉTER
    return w_std, b_std

In [ ]:
# Deux versions de la même règle : distance en km, puis en mètres
X_km = t2_sud[['surface_m2', 'distance_km']].to_numpy(dtype=float)
X_m = X_km * np.array([1, 1000])
w_km, w_m, b = np.array([3, -8]), np.array([3, -0.008]), 45
print('prédictions (km) :', X_km @ w_km + b)
print('prédictions (m)  :', X_m @ w_m + b)
resultats = {}
for nom, Xv, wv in [('km', X_km, w_km), ('m', X_m, w_m)]:
    mu, sigma = Xv.mean(axis=0), Xv.std(axis=0, ddof=0)
    resultats[nom] = poids_standardises(wv, b, mu, sigma)
    print(nom, ': moyenne distance =', mu[1], ', écart-type =', sigma[1], ', poids standardisés =', resultats[nom])

In [ ]:
# Vérification
if resultats['km'][0] is None:
    print('Complétez la fonction poids_standardises.')
else:
    ok = all(np.allclose(resultats[k][0], [12, -8]) and np.isclose(resultats[k][1], 164) for k in resultats)
    print('poids standardisés identiques en km et en m :', 'correct' if ok else 'à revoir')

In [ ]:
if resultats['km'][0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
    for ax, titre, valeurs in [
            (axes[0], 'Poids bruts du 2e terme', {'km': w_km[1], 'm': w_m[1]}),
            (axes[1], 'Poids standardisés du 2e terme', {k: resultats[k][0][1] for k in resultats})]:
        ax.bar(list(valeurs), list(valeurs.values()), color=[BLEU, ORANGE])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(titre, fontsize=10)
    plt.tight_layout()
    plt.show()

*Votre conclusion pour c) :* …

## Exercice 2.6 — Défi : une règle, plusieurs écritures

*Données D2.4. Notions : leçons 2.3, 2.4 et 2.5.* Trois colonnes indicatrices : Parc, Rocade, Gare.

In [ ]:
gare = (est['quartier'] == 'Gare').astype(int).to_numpy()
parc3 = (est['quartier'] == 'Parc').astype(int).to_numpy()
rocade3 = (est['quartier'] == 'Rocade').astype(int).to_numpy()
X3 = np.column_stack([est['surface_m2'], est['distance_km'], parc3, rocade3, gare])
print('somme des trois indicatrices, ligne par ligne :', X3[:, 2:].sum(axis=1))
print('règle du collègue (b = 40) :', X3 @ np.array([3, -8, 17, -5, 5]) + 40)

In [ ]:
# À COMPLÉTER b) : trois poids de quartier (Parc, Rocade, Gare) pour b = 50
poids_quartier_b50 = None   # liste de trois nombres

In [ ]:
# Vérification
if poids_quartier_b50 is None:
    print('Complétez la cellule précédente.')
else:
    y50 = X3 @ np.array([3, -8] + list(poids_quartier_b50)) + 50
    print('prédictions avec b = 50 :', y50, '→', 'correct' if np.allclose(y50, [221, 199, 209, 184, 251]) else 'à revoir')

*Votre réponse à c) :* …